# Metrics

In [98]:
from google.cloud import bigquery
import gender_guesser.detector as gender
import pandas as pd
from df2gspread import gspread2df as g2d

from tqdm import tqdm
tqdm.pandas()

import warnings
warnings.filterwarnings("ignore")

## Class metrics

In [2]:
class MetricsComputation:
    
    def __init__(self, inst_ids_list, PROJECT_ID, DATASET_ID, cerca_centers, cerca_oa_ids):
        self.inst_ids = list(set(sum(list(inst_ids_list.values()), [])))
        self.GBQ_PROJECT_ID = PROJECT_ID
        self.GBQ_DATASET_ID = DATASET_ID
        self.gend = gender.Detector()
        self.cerca_centers = cerca_centers
        self.cerca_oa_ids = cerca_oa_ids
        
    def bg_query(self, query):
        client = bigquery.Client(project = self.GBQ_PROJECT_ID)
        df = client.query(query)
        return df.to_dataframe()
    
    def gender_detection(self):
        df_gender = self.df_inst.drop_duplicates(subset = ['display_name']).reset_index(drop = True)
        df_gender['gender'] = df_gender.first_name.progress_apply(lambda x: self.gend.get_gender(x))
        self.df_inst = self.df_inst.merge(df_gender[['display_name', 'gender']], on = 'display_name', how = 'left')
        return self.df_inst
        
    def get_institution_data(self):
        institutions_sql = "(" + ",".join(f"'{i}'" for i in self.inst_ids) + ")"

        sql = f"""SELECT ww.DOI, a.display_name, wa.author_order, wa.author_position, wa.is_corresponding, wa.INSTITUTION_ID, wins.COUNTRY_CODE 
                    FROM `{self.GBQ_PROJECT_ID}.{self.GBQ_DATASET_ID}.works` ww
                    JOIN `{self.GBQ_PROJECT_ID}.{self.GBQ_DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID AND CAST(wa.INSTITUTION_ID AS STRING) IN {institutions_sql}
                    JOIN `{self.GBQ_PROJECT_ID}.{self.GBQ_DATASET_ID}.authors` a ON a.ID = wa.author_id
                    LEFT JOIN `{self.GBQ_PROJECT_ID}.{self.GBQ_DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
                    WHERE ww.PUBLICATION_YEAR BETWEEN 2021 AND 2024"""
                    
        self.df_inst = self.bg_query(sql).dropna(subset = ['DOI']).reset_index(drop = True)
        self.df_inst['first_name'] = self.df_inst['display_name'].str.split(' ').str[0]
        self.df_inst['gender'] = self.df_inst.first_name.progress_apply(lambda x: self.gend.get_gender(x))
        
        return self.df_inst
    
    def get_institution_collaboration(self):
        in_query = str(tuple(self.df_inst.DOI.unique().tolist()))
        in_query = in_query.replace(',)', ')')

        sql = f"""SELECT ww.DOI, wa.INSTITUTION_ID, wins.COUNTRY_CODE
                    FROM `{self.GBQ_PROJECT_ID}.{self.GBQ_DATASET_ID}.works` ww
                    JOIN `{self.GBQ_PROJECT_ID}.{self.GBQ_DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
                    JOIN `{self.GBQ_PROJECT_ID}.{self.GBQ_DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
                    WHERE ww.PUBLICATION_YEAR BETWEEN 2021 AND 2024 AND ww.DOI IN {in_query}
                """
        self.df_inst_colab = self.bg_query(sql).drop_duplicates().reset_index(drop = True)
        return self.df_inst_colab
        
    def metrics(self):
        
        self.df_inst = self.get_institution_data()
        self.df_inst.to_csv('../data/interim/institution_publications_%s.csv'%self.inst_ids, index = False)
        
        pubs_number = self.df_inst.DOI.nunique()
        
        df_led_pubs = self.df_inst[(self.df_inst.author_position == 'first') | (self.df_inst.author_position == 'last') | (self.df_inst.is_corresponding == True)]
        per_leds_pubs = df_led_pubs.DOI.nunique() / pubs_number

        df_fem_pubs = self.df_inst[self.df_inst.gender == 'female']
        per_fem_pub = df_fem_pubs.DOI.nunique() / pubs_number
        
        df_led_fem_pubs = df_fem_pubs[(df_fem_pubs.author_position == 'first') | (df_fem_pubs.author_position == 'last') | (df_fem_pubs.is_corresponding == True)]
        per_led_fem_pubs = df_led_fem_pubs.DOI.nunique() / pubs_number
        
        self.df_inst_colab = self.get_institution_collaboration()
        
        df_cerca = self.df_inst_colab[(self.df_inst_colab.INSTITUTION_ID.isin(self.cerca_oa_ids)) & (~self.df_inst_colab.INSTITUTION_ID.isin([int(x) for x in self.inst_ids]))]
        per_cerca_pubs = df_cerca.DOI.nunique() / pubs_number
        
        cerca_inst = list(set(sum(list(self.cerca_centers.values()), [])))
        cerca_inst = [int(x) for x in cerca_inst if isinstance(x, (str, int)) and str(x).isdigit()]
        if any(x in cerca_inst for x in [int(x) for x in self.inst_ids]):
            df_not_cerca_colab = self.df_inst_colab[(~self.df_inst_colab.DOI.isin(df_cerca.DOI)) & (self.df_inst_colab.COUNTRY_CODE == 'ES')]
            # df_non_cerca_esp = self.df_inst_colab[(~self.df_inst_colab.INSTITUTION_ID.isin(self.cerca_oa_ids)) & (self.df_inst_colab.COUNTRY_CODE == self.df_inst.COUNTRY_CODE.dropna().unique()[0])]
            print(f"The percentage of publications in collaboration for {self.inst_ids} is: {df_not_cerca_colab.DOI.nunique() / pubs_number :.2%}")
        
        try:
            df_inter = self.df_inst_colab[self.df_inst_colab.COUNTRY_CODE != self.df_inst.COUNTRY_CODE.dropna().unique()[0]]
        except:
            print('fucking exception!!!!!!!!!!!!!!!!!')
            country_counts = self.df_inst_colab.groupby('DOI')['COUNTRY_CODE'].nunique()
            single_country_dois = country_counts[country_counts == 1].index
            print('fucking exception!!!!!!!!!!!!!!!!!')
            df_inter = self.df_inst_colab[~self.df_inst_colab['DOI'].isin(single_country_dois)]
        per_inter_pubs = df_inter.DOI.nunique() / pubs_number

        return pubs_number, per_leds_pubs, per_fem_pub, per_led_fem_pubs, per_cerca_pubs, per_inter_pubs
        

In [4]:
gbq_project = 'siris-datasets'
gbq_dataset = 'openalex'

cerca_centers = {'CRM' : ['4210122226'],
                 'ICRA' : ['2799562678'],
                 'CTFC' : ['4210117018']}

interest_centers = {'CRM' : ['4210122226'],
                    'ICRA' : ['2799562678'],
                    'CTFC' : ['4210117018'],
                    'IMT' : ['84500057'],
                    'HCM' : ['4391767997'],
                    'MPI-MIS' : ['4210091327'],
                    'ICMAT' : ['4210157032'],                 
                    # 'WETSUS' : ['4413047491'], THE WETSUS IS NOT RETURNING PUBLICATIONS !
                    'NIVA' : ['2802939387'],
                    'KWR Water Research Institute' : ['4210139073']}

cerca_af = list(pd.read_csv('../data/external/ToCheck - AffID.csv').OA_id)

results = []
for name, ids in tqdm(interest_centers.items(), total = len(interest_centers), desc = "Processing centers"):
    tqdm.write(f"Processing: {name}")
    center_dict = {name: ids}
    metrics_obj = MetricsComputation(center_dict, gbq_project, gbq_dataset, cerca_centers, cerca_af)
    results.append(metrics_obj.metrics())
df_results = pd.DataFrame(results, interest_centers.keys(), columns = ['# Pubs', '% led Pubs', '% Fem Pubs', '% Fem led Pubs', '% CERCA Colab Pubs', '% Inter Colab Pubs'])
df_results.to_csv('../data/processed/benchmark_results_segonaentrega.csv')
df_results

Processing centers:   0%|          | 0/9 [00:00<?, ?it/s]

Processing: CRM


Processing centers:  11%|█         | 1/9 [00:07<00:59,  7.41s/it]

The percentage of publications in collaboration for ['4210122226'] is: 85.24%
Processing: ICRA


Processing centers:  22%|██▏       | 2/9 [00:15<00:54,  7.79s/it]

...

Processing: NIVA


Processing centers:  89%|████████▉ | 8/9 [00:55<00:06,  6.78s/it]

Processing: KWR Water Research Institute


Processing centers: 100%|██████████| 9/9 [01:01<00:00,  6.87s/it]


,# Pubs,% led Pubs,% Fem Pubs,% Fem led Pubs,% CERCA Colab Pubs,% Inter Colab Pubs
CRM,542,0.719557,0.274908,0.184502,0.147601,0.669742
ICRA,761,0.536137,0.438896,0.143233,0.232589,0.579501
CTFC,555,0.547748,0.380180,0.158559,0.209009,0.664865
IMT,1336,0.743263,0.161677,0.089072,0.005240,0.481287
HCM,60,0.766667,0.116667,0.066667,0.016667,0.650000
MPI-MIS,823,0.777643,0.131227,0.095990,0.002430,0.739976
ICMAT,741,0.798920,0.143050,0.087719,0.020243,0.650472
NIVA,960,0.507292,0.529167,0.225000,0.022917,0.721875
KWR Water Research Institute,531,0.508475,0.282486,0.143126,0.016949,0.647834


## DOIS based computation FOR WETSUS

**For the ICRA which has also passed DOIS**

In [90]:
interest_centers = ['ICRA']
file_path = '../data/external/5_Bibliometria_SIRIS_031125/'

center_name = 'ICRA/'
file_name = 'DOIS_ICRA'

interest_centers = {'ICRA' : ['2799562678'],                 
                    'WETSUS' : ['4413047491'],
                    'NIVA' : ['2802939387'],
                    'KWR' : ['4210139073']}

df_WETSUS = pd.read_csv(file_path + 'BM_' + center_name + file_name + '.csv')[['WETSUS']].dropna().reset_index(drop = True)
df_WETSUS

,WETSUS
0,10.1016/j.memsci.2020.118538
1,10.1016/j.jcis.2020.07.146
2,10.1111/1462-2920.15311
3,10.1016/j.tibtech.2020.06.006
4,10.1039/d0ew00203h
...,...
285,10.1186/s13705-024-00485-w
286,10.3390/toxics12120889
287,10.1038/s41467-024-45758-2
288,10.1016/j.biortech.2024.131529


In [94]:
def bg_query( query):
    client = bigquery.Client(project = gbq_project)
    df = client.query(query)
    return df.to_dataframe()

PROJECT_ID = 'siris-datasets'
DATASET_ID = 'openalex'

in_query = str(tuple(df_WETSUS.WETSUS.str.lower().tolist())) # IMPORTANT TO LOWER THEM!
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
        a.display_name,
        wa.author_order,
        wa.author_position,
        wa.is_corresponding,
        wins.ID AS institution_id,
        war.raw_affiliation,
        wins.COUNTRY_CODE
        FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
        LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
        LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
        LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
        LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships_raw` war ON war.WORK_ID = ww.ID AND war.AUTHOR_ID = wa.AUTHOR_ID
        WHERE ww.DOI IN {in_query}
        """
    
df_inst = bg_query(sql).dropna(subset = ['DOI']).reset_index(drop = True)  
df_inst

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,raw_affiliation,COUNTRY_CODE
0,10.1016/j.cub.2023.08.026,Alessia Guggisberg,36,middle,False,35440088,"ETH Zürich, Institut für Integrative Biologie,...",CH
1,10.1016/j.cej.2022.138412,Catarina Simões,1,first,True,94624287,"Sustainable Process Technology, Faculty of Sci...",NL
2,10.1016/j.cej.2022.138412,Catarina Simões,1,first,True,94624287,"Wetsus, European Centre of Excellence for Sust...",NL
3,10.1021/acs.est.2c02686,Lorrie Maccario,3,middle,False,124055696,"Department of Biology, University of Copenhage...",DK
4,10.1016/j.soilbio.2022.108831,Valentina Sechi,4,middle,False,<NA>,"Wetsus, European Centre of Excellence for Sust...",None
...,...,...,...,...,...,...,...,...
2726,10.1016/j.isci.2021.102095,Marcel Dickmann,6,middle,False,4210097247,"Heinz Maier-Leibnitz Zentrum (MLZ), Technische...",DE
2727,10.1016/j.isci.2021.102095,Marcel Dickmann,6,middle,False,62916508,"Heinz Maier-Leibnitz Zentrum (MLZ), Technische...",DE
2728,10.1016/j.jwpe.2024.104932,Javier A. Pavez-Jara,1,first,True,98358874,"Department of Water Management, Delft Universi...",NL
2729,10.1007/s11269-021-02768-9,Doekle Yntema,2,middle,False,<NA>,"Wetsus, European Centre of Excellence for Sust...",None


In [96]:
grouped = df_inst.groupby('raw_affiliation').count().sort_values('DOI', ascending = False)[['DOI']]
grouped.to_csv('to_check_WETSUS.csv')
grouped

,DOI
raw_affiliation,
"Wetsus, European Centre of Excellence for Sustainable Water Technology, Oostergoweg 9, 8911 MA Leeuwarden, The Netherlands",76
"Wetsus, European Centre of Excellence for Sustainable Water Technology, Oostergoweg 9, 8911 MA Leeuwarden, the Netherlands",50
"Wetsus, European Centre of Excellence for Sustainable Water Technology, Leeuwarden, The Netherlands",44
"Wetsus, European Centre of Excellence for Sustainable Water Technology, Oostergoweg 9, 8911 MA, Leeuwarden, The Netherlands",25
"Department of Biotechnology, Delft University of Technology, Van der Maasweg 9, 2629 HZ Delft, The Netherlands",25
...,...
"Wetsus, European Centre of Excellence for Sustainable Water, Oostergoweg 9, 8911, MA, Leeuwarden, the Netherlands",1
"Wetsus, Leeuwarden, Friesland, The Netherlands",1
"Working Group Metrology–Laser Optical Metrology, Institute for Thermal Turbomachinery and Machine Dynamics, Graz University of Technology, Inffeldgasse 25A, 8010 Graz, Austria",1


In [ ]:
df_check = g2d.download('1J7uywXX7fsxjNbUSi24uQJiW5bTjbKHba1RjqdYBwQA', 'WETSUS', col_names = True, row_names = False)
wetsus = list(df_check[df_check.WETSUS == 'TRUE'].raw_affiliation)
df_inst['WETSUS'] = df_inst.raw_affiliation.apply(lambda x: True if x in wetsus else False)
df_inst

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,raw_affiliation,COUNTRY_CODE,WETSUS
0,10.1016/j.cub.2023.08.026,Alessia Guggisberg,36,middle,False,35440088,"ETH Zürich, Institut für Integrative Biologie,...",CH,False
1,10.1016/j.cej.2022.138412,Catarina Simões,1,first,True,94624287,"Sustainable Process Technology, Faculty of Sci...",NL,False
2,10.1016/j.cej.2022.138412,Catarina Simões,1,first,True,94624287,"Wetsus, European Centre of Excellence for Sust...",NL,True
3,10.1021/acs.est.2c02686,Lorrie Maccario,3,middle,False,124055696,"Department of Biology, University of Copenhage...",DK,False
4,10.1016/j.soilbio.2022.108831,Valentina Sechi,4,middle,False,<NA>,"Wetsus, European Centre of Excellence for Sust...",None,True
...,...,...,...,...,...,...,...,...,...
2726,10.1016/j.isci.2021.102095,Marcel Dickmann,6,middle,False,4210097247,"Heinz Maier-Leibnitz Zentrum (MLZ), Technische...",DE,False
2727,10.1016/j.isci.2021.102095,Marcel Dickmann,6,middle,False,62916508,"Heinz Maier-Leibnitz Zentrum (MLZ), Technische...",DE,False
2728,10.1016/j.jwpe.2024.104932,Javier A. Pavez-Jara,1,first,True,98358874,"Department of Water Management, Delft Universi...",NL,False
2729,10.1007/s11269-021-02768-9,Doekle Yntema,2,middle,False,<NA>,"Wetsus, European Centre of Excellence for Sust...",None,True


**% led publications**

In [107]:
df_wetsus= df_inst[df_inst.WETSUS == True]
df_led = df_wetsus[(df_wetsus.author_position == 'first') | (df_wetsus.author_position == 'last') |(df_wetsus.is_corresponding == True) ]
df_led.DOI.nunique() / df_inst.DOI.nunique()

0.6816608996539792

**\% publications with women from the centre as authors**

In [108]:
gend = gender.Detector()

df_wetsus['first_name'] = df_wetsus['display_name'].str.split(' ').str[0]
df_wetsus['gender'] = df_wetsus.first_name.progress_apply(lambda x: gend.get_gender(x))

df_wetsus

100%|██████████| 758/758 [00:00<00:00, 341711.35it/s]


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,raw_affiliation,COUNTRY_CODE,WETSUS,first_name,gender
2,10.1016/j.cej.2022.138412,Catarina Simões,1,first,True,94624287,"Wetsus, European Centre of Excellence for Sust...",NL,True,Catarina,female
4,10.1016/j.soilbio.2022.108831,Valentina Sechi,4,middle,False,<NA>,"Wetsus, European Centre of Excellence for Sust...",None,True,Valentina,female
15,10.1016/j.desal.2023.116580,S. Porada,4,middle,False,11923345,"Wetsus, European Centre of Excellence for Sust...",PL,True,S.,unknown
17,10.1016/j.desal.2023.116580,S. Porada,4,middle,False,686019,"Wetsus, European Centre of Excellence for Sust...",PL,True,S.,unknown
27,10.1016/j.watres.2024.122407,W.K. Wijdeveld,3,middle,False,<NA>,"Wetsus, European Centre of Excellence for Sust...",None,True,W.K.,unknown
...,...,...,...,...,...,...,...,...,...,...,...
2716,10.1016/j.jil.2023.100058,Hardy Temmink,3,middle,False,913481162,Wetsus – European Centre of Excellence for Sus...,NL,True,Hardy,male
2717,10.3390/w15122167,Wiecher Bakx,1,first,True,193662353,"Wetsus, European Centre of Excellence for Sust...",NL,True,Wiecher,unknown
2720,10.1016/j.jwpe.2023.104648,Roel J.W. Meulepas,3,middle,False,<NA>,"Wetsus, European Centre of Excellence for Sust...",None,True,Roel,male
2722,10.1016/j.desal.2021.115091,P.A. Sosa-Fernandez,1,first,False,913481162,"Wetsus, European centre of excellence for sust...",NL,True,P.A.,unknown


In [109]:
df_fem = df_wetsus[df_wetsus.gender.isin(['female'])]

df_fem.DOI.nunique() / df_inst.DOI.nunique()

0.39792387543252594

**\% publications led by women from the centre as authors**

In [110]:
df_fem_led = df_fem[(df_fem.author_position == 'first') | (df_fem.author_position == 'last') |(df_fem.is_corresponding == True) ]

df_fem_led.DOI.nunique() / df_inst.DOI.nunique()

0.19377162629757785

**\% publications in collaboration with other CERCA centres**

In [113]:
df_cerca_af = pd.read_csv('../data/external/ToCheck - AffID.csv')

df_center = df_inst[df_inst.WETSUS != True]
df_colab = df_center[df_center.institution_id.isin(df_cerca_af.OA_id)] 
df_colab.DOI.nunique() / df_inst.DOI.nunique()

0.006920415224913495

**\% publications in collaboration with other international institutions**

In [122]:
df_international_non_cerca = df_center[df_center['COUNTRY_CODE'] != 'NL']

dois_center = set(df_wetsus.DOI)
dois_international = set(df_international_non_cerca['DOI'])

collaborative_dois = dois_center & dois_international
    
len(collaborative_dois) / df_inst.DOI.nunique()

0.4463667820069204